# 🏷 chat-template の実体

`--chat-template` の定義と .jinja 自動解決の有無。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")

import re

print("=== chat_template の定義箇所 ===")
for f in sorted(root.rglob("*.py")):
    t = f.read_text(errors="replace")
    if "chat_template" not in t:
        continue
    for i, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if "chat_template" in s and ("add_argument" in s or "kwargs[" in s
                                     or "chat_template:" in s
                                     or "chat_template =" in s
                                     or "chat_template.jinja" in s):
            print(f"  {f.relative_to(root)}:{i}: {s[:160]}")

print("\n=== ChatTemplateConfig（あれば）===")
for f in sorted(root.rglob("*chat_template*.py")):
    print("  file:", f.relative_to(root))
    t = f.read_text(errors="replace")
    for i, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if s.startswith("class ") or s.startswith("def "):
            print(f"    {i}: {s[:140]}")

print("\n=== tokenizer 側の解決（tokenizer_config に無い場合）===")
for f in sorted(root.rglob("transformers_utils/*.py")) + sorted(root.rglob("entrypoints/**/*.py")):
    t = f.read_text(errors="replace")
    if "chat_template.jinja" in t or "chat_template.json" in t:
        for i, l in enumerate(t.splitlines(), 1):
            if "chat_template" in l:
                print(f"  {f.relative_to(root)}:{i}: {l.strip()[:160]}")

print("\n=== get_chat_template / load_chat_template 実装 ===")
for f in sorted(root.rglob("*.py")):
    t = f.read_text(errors="replace")
    for m in re.finditer(r"def (get_chat_template|load_chat_template|"
                         r"resolve_chat_template)[^(]*\\(", t):
        i = m.start()
        print(f"  --- {f.relative_to(root)} ---")
        print(t[i:i + 1200])
        break

print("\n=== API リクエスト時の chat_template_kwargs 対応 ===")
for f in sorted(root.rglob("*.py")):
    t = f.read_text(errors="replace")
    if "chat_template_kwargs" in t:
        n = t.count("chat_template_kwargs")
        print(f"  {f.relative_to(root)}: {n}")
